# Building Multimodal RAG Application — Part 2: Multimodal Embeddings

This Colab-ready companion notebook reproduces the article workflow with the current open-source Hugging Face BridgeTower interface. It creates joint image-text embeddings, compares them with cosine similarity and Euclidean distance, and optionally projects a larger sample into two dimensions with UMAP.

The original October 2024 article used a hosted Prediction Guard helper. This notebook uses the official `BridgeTower/bridgetower-large-itm-mlm-itc` checkpoint directly so no third-party API key is required.

- [BridgeTower model card](https://huggingface.co/BridgeTower/bridgetower-large-itm-mlm-itc)
- [Transformers BridgeTower documentation](https://huggingface.co/docs/transformers/model_doc/bridgetower)
- [BridgeTower paper](https://arxiv.org/abs/2206.08657)

> The checkpoint is several gigabytes. Use a Colab GPU runtime and allow time for the first download.


In [ ]:
!pip -q install "transformers>=4.48,<6" torch pillow requests numpy pandas scikit-learn umap-learn matplotlib seaborn datasets tqdm


In [ ]:
from io import BytesIO
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from datasets import load_dataset
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
from transformers import BridgeTowerForContrastiveLearning, BridgeTowerProcessor
from umap import UMAP

MODEL_ID = "BridgeTower/bridgetower-large-itm-mlm-itc"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = BridgeTowerProcessor.from_pretrained(MODEL_ID)
model = BridgeTowerForContrastiveLearning.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"Using {DEVICE}")


## 1. Load three image-text examples

In [ ]:
examples = [
    {
        "url": "https://farm3.staticflickr.com/2519/4126738647_cc436c111b_z.jpg",
        "caption": "A motorcycle sits parked across from a herd of livestock",
    },
    {
        "url": "https://farm3.staticflickr.com/2046/2003879022_1b4b466d1d_z.jpg",
        "caption": "Motorcycle on a platform to be worked on in a garage",
    },
    {
        "url": "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
        "caption": "A dog sitting outdoors",
    },
]


def load_image(url: str) -> Image.Image:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


for example in examples:
    example["image"] = load_image(example["url"])
    display(example["image"])
    print(example["caption"])


## 2. Compute joint BridgeTower embeddings

In [ ]:
@torch.inference_mode()
def joint_embedding(image: Image.Image, text: str) -> np.ndarray:
    inputs = processor(images=image, text=text, return_tensors="pt")
    inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
    output = model(**inputs)
    embedding = output.cross_embeds[0]
    embedding = torch.nn.functional.normalize(embedding, dim=-1)
    return embedding.cpu().numpy()


embeddings = np.stack(
    [joint_embedding(example["image"], example["caption"]) for example in examples]
)
print("Embedding matrix shape:", embeddings.shape)


## 3. Compare embeddings

In [ ]:
def cosine_similarity(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(np.dot(vector_a, vector_b) / (np.linalg.norm(vector_a) * np.linalg.norm(vector_b)))


pairs = [(0, 1), (1, 2), (0, 2)]
for left, right in pairs:
    cosine = cosine_similarity(embeddings[left], embeddings[right])
    euclidean = float(np.linalg.norm(embeddings[left] - embeddings[right]))
    print(f"Examples {left + 1} and {right + 1}: cosine={cosine:.4f}, euclidean={euclidean:.4f}")


The two motorcycle pairs should generally be closer to each other than either is to the cat pair. Exact values can vary with library and checkpoint revisions.

## 4. Optional larger UMAP visualization

The following section downloads cat and car datasets and computes many BridgeTower embeddings. It is intentionally opt-in because the model is large and this step can take several minutes even on a GPU.


In [ ]:
TEMPLATES = [
    "an image of a {}",
    "a photo of a {}",
    "a close-up photo of a {}",
]


def prepare_dataset_for_umap_visualization(
    dataset_name: str,
    class_name: str,
    sample_size: int = 50,
):
    dataset = load_dataset(dataset_name, split=f"train[:{sample_size}]")
    return [
        {
            "caption": random.choice(TEMPLATES).format(class_name),
            "image": row["image"].convert("RGB"),
        }
        for row in dataset
    ]


RUN_LARGE_DEMO = False
SAMPLE_SIZE = 50


In [ ]:
if RUN_LARGE_DEMO:
    cat_pairs = prepare_dataset_for_umap_visualization(
        "yashikota/cat-image-dataset", "cat", SAMPLE_SIZE
    )
    car_pairs = prepare_dataset_for_umap_visualization(
        "tanganke/stanford_cars", "car", SAMPLE_SIZE
    )

    cat_embeddings = np.stack(
        [joint_embedding(pair["image"], pair["caption"]) for pair in tqdm(cat_pairs)]
    )
    car_embeddings = np.stack(
        [joint_embedding(pair["image"], pair["caption"]) for pair in tqdm(car_pairs)]
    )

    all_embeddings = np.concatenate([cat_embeddings, car_embeddings])
    labels = ["cat"] * len(cat_embeddings) + ["car"] * len(car_embeddings)
else:
    print("Set RUN_LARGE_DEMO = True and rerun this cell to compute the larger sample.")


In [ ]:
def dimensionality_reduction(embedding_array: np.ndarray, labels: list[str]) -> pd.DataFrame:
    scaled = MinMaxScaler().fit_transform(embedding_array)
    mapper = UMAP(n_components=2, metric="cosine", random_state=42).fit(scaled)
    frame = pd.DataFrame(mapper.embedding_, columns=["X", "Y"])
    frame["label"] = labels
    return frame


if RUN_LARGE_DEMO:
    reduced_embeddings = dimensionality_reduction(all_embeddings, labels)
    figure, axis = plt.subplots(figsize=(8, 6))
    sns.set_style("whitegrid", {"axes.grid": False})
    sns.scatterplot(data=reduced_embeddings, x="X", y="Y", hue="label", palette="bright", ax=axis)
    sns.move_legend(axis, "upper left", bbox_to_anchor=(1, 1))
    axis.set_title("BridgeTower embeddings of cats and cars projected with UMAP")
    plt.tight_layout()
    plt.show()


## What to try next

- Replace the captions while keeping the images fixed and compare how the joint embeddings move.
- Add another visual category and inspect whether UMAP separates it.
- Store the normalized embeddings in a vector database for the retrieval stage of the multimodal RAG pipeline.
